# 02 — Non-IID Partition Visualization
Shows how the Dirichlet(α=0.5) partitioning distributes attack classes across 50 clients.

In [ ]:
import sys, pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parent))
from src.configs.paths import DATA_DIR, PREPROCESSED_DIR, ARTIFACTS_DIR
from src.configs.config import CONFIG

with open(PREPROCESSED_DIR / 'label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)

num_clients = CONFIG['federated']['num_clients']
num_classes = CONFIG['model']['num_classes']
class_names = le.classes_

# Build class count matrix: shape (num_clients, num_classes)
matrix = np.zeros((num_clients, num_classes), dtype=int)
for cid in range(num_clients):
    path = DATA_DIR / f'client_{cid:04d}.npz'
    if not path.exists():
        continue
    d = np.load(path)
    y = d['y'] if 'y' in d else d.get('y_train', np.array([]))
    for cls in range(num_classes):
        matrix[cid, cls] = (y == cls).sum()

print(f'Loaded {num_clients} client partitions.')
print(f'Total samples across all clients: {matrix.sum():,}')

In [ ]:
# Class distribution heatmap across all clients
fig, ax = plt.subplots(figsize=(20, 12))
df_heat = np.log1p(matrix)  # log scale for visibility
sns.heatmap(df_heat, cmap='YlOrRd', ax=ax, linewidths=0.1,
            xticklabels=class_names, yticklabels=[f'C{i}' for i in range(num_clients)])
ax.set_xlabel('Attack Class')
ax.set_ylabel('Client ID')
ax.set_title(f'Non-IID Partition (Dirichlet α={CONFIG["data"]["alpha_dirichlet"]}) — log(1+count)')
plt.xticks(rotation=60, ha='right', fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'plots' / 'noniid_partition_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Samples per client (total)
client_totals = matrix.sum(axis=1)
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(num_clients), client_totals, color='teal', edgecolor='white')
ax.axhline(client_totals.mean(), color='red', linestyle='--', label=f'Mean={client_totals.mean():.0f}')
ax.set_xlabel('Client ID')
ax.set_ylabel('Total training samples')
ax.set_title('Training samples per client')
ax.legend()
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'plots' / 'samples_per_client.png', dpi=150)
plt.show()
print(f'Min: {client_totals.min():,}  Max: {client_totals.max():,}  Std: {client_totals.std():.0f}')

In [ ]:
# Classes with zero coverage (clients that never see a class)
zero_coverage = (matrix == 0).sum(axis=0)
print('Classes with most zero-coverage clients (top 10):')
for cls_idx in np.argsort(zero_coverage)[::-1][:10]:
    print(f'  {class_names[cls_idx]:30s}: {zero_coverage[cls_idx]}/{num_clients} clients have zero samples')